# AF2 spectral — staged global decision
Attach the private core dataset plus the **Saved Version outputs** from Stage-1, PCG1, and WAV1. No local download is required. This notebook reads the seven seed-42 result JSONs directly from `/kaggle/input`, safely deduplicates identical copies, reconstructs Stage-1/Stage-2/global frozen decisions, and never trains or opens test data. ZIP fallback is still supported.


In [ ]:
import hashlib, importlib, json, os, shutil, subprocess, sys, zipfile
from pathlib import Path
WORK=Path('/kaggle/working'); INPUT=Path('/kaggle/input'); REPO=WORK/'coffee-bean-detection'; OUT=WORK/'af2-spectral-factorization-v1'; REPORTS=OUT/'val_reports'
os.chdir(WORK)
if REPO.exists(): shutil.rmtree(REPO)
for attempt in range(3):
    result=subprocess.run(['git','clone','--depth','1','--branch','agent/af2-spectral-factorization','https://github.com/ediprin/coffee-bean-detection.git',str(REPO)])
    if result.returncode==0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt==2: raise RuntimeError('git clone gagal tiga kali')
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(REPO)],check=True)
for module_name in list(sys.modules):
    if module_name=='coffee_detector' or module_name.startswith('coffee_detector.'): sys.modules.pop(module_name,None)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
print('COMMIT:',subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip())
REPORTS.mkdir(parents=True,exist_ok=True)

def unique_equivalent(paths,label):
    paths=[p for p in paths if p.is_file()]
    if not paths: return None
    groups={}
    for p in paths:
        h=hashlib.sha256(p.read_bytes()).hexdigest(); groups.setdefault(h,[]).append(p)
    if len(groups)!=1:
        detail={h:[str(p) for p in ps] for h,ps in groups.items()}
        raise RuntimeError(f'{label} memiliki beberapa isi berbeda: {detail}')
    chosen=sorted(next(iter(groups.values())))[0]
    if len(paths)>1: print(f'{label}: {len(paths)} salinan identik; memakai {chosen}')
    return chosen

arms=('AF2WIN','AF2ORI','AF2POL','AF2SOFT','AF2LUM','PCG1','WAV1')
sources={arm:unique_equivalent(sorted(INPUT.rglob(f'{arm}_seed42_result.json')),arm) for arm in arms}
# Optional fallback for legacy ZIP handoffs.
missing=[arm for arm,src in sources.items() if src is None]
if missing:
    EXTRACT=WORK/'af2-spectral-global-input'; shutil.rmtree(EXTRACT,ignore_errors=True); EXTRACT.mkdir(parents=True)
    for name in ('af2-spectral-stage1-sequential-output.zip','PCG1_seed42_output.zip','WAV1_seed42_output.zip'):
        matches=sorted(p for p in INPUT.rglob(name) if p.is_file())
        if len(matches)>1: raise RuntimeError(f'{name} ambigu: {matches}')
        if len(matches)==1:
            dst=EXTRACT/Path(name).stem; dst.mkdir(parents=True)
            with zipfile.ZipFile(matches[0],'r') as handle: handle.extractall(dst)
    for arm in missing:
        sources[arm]=unique_equivalent(sorted(EXTRACT.rglob(f'{arm}_seed42_result.json')),f'ZIP {arm}')
for arm,src in sources.items():
    if src is None: raise FileNotFoundError(f'Hasil {arm} tidak ditemukan. Attach Saved Version output Stage-1, PCG1, dan WAV1.')
    payload=json.loads(src.read_text(encoding='utf-8'))
    assert payload['arm']==arm and payload['seed']==42 and payload['evaluation_split']=='val' and payload['test_images_accessed'] is False
    shutil.copy2(src,REPORTS/src.name); print('RESULT READY:',arm,src)
baseline=unique_equivalent(sorted(INPUT.rglob('lfdet_afab_seed42_screening.json')),'AF2 baseline')
if baseline is None: raise FileNotFoundError('Harus attach private core dataset yang memuat evidence AF2')
from coffee_detector.experiments.run_faruq_v3_af2_spectral_decision import run_spectral_decision
stage1=run_spectral_decision(OUT,baseline,stage='stage1'); assert stage1['test_opened'] is False and stage1['next']=='AUTHORIZE_STAGE2'
stage2=run_spectral_decision(OUT,baseline,stage='stage2'); assert stage2['test_opened'] is False and stage2['next']=='AUTHORIZE_GLOBAL_DECISION'
global_result=run_spectral_decision(OUT,baseline,stage='global'); assert global_result['test_opened'] is False
print('=== GLOBAL RESULT ==='); print(json.dumps(global_result,indent=2))
small=WORK/'af2-spectral-global-decision.json'; small.write_text(json.dumps(global_result,indent=2)+'\n',encoding='utf-8')
print('GLOBAL DECISION SAVED:',small)
print('Gunakan Save Version. Tidak perlu download checkpoint/output besar.')
